<a href="https://colab.research.google.com/github/Anashrah/Anashrah_Adil_ML_ITAI1371_12321_FINAL/blob/main/Anashrah_Adil_ML_ITAI1371_12321_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stroke Prediction Dataset

https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset?resource=download




## 1. Preproccess the Data

Splitting our data into Train (70%), Validation (15%), and Test (15%)

The midterm highlights that the stroke dataset is heavily imbalanced (~95% no stroke vs. ~5% stroke). To fix this without corrupting our evaluation, and we must apply SMOTE (Synthetic Minority Over-sampling Technique) correctly:



In [ ]:
!pip install imbalanced-learn -q

from google.colab import files
uploaded = files.upload()


Saving healthcare-dataset-stroke-data (1).csv to healthcare-dataset-stroke-data (1) (1).csv


In [ ]:
import io
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset from the uploaded dictionary
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Drop missing values or handle imputations
df = df.dropna()

# 2. Separating Features (X) and Target Label (y)
X = df.drop(columns=['stroke', 'id'])  # 'id' is an identifier, not a feature
y = df['stroke']

# 3. First Split: Separate out the Test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# 4. Second Split: Separate remaining data into Train (~70% total) and Validation (~15% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.17647, random_state=42, stratify=y_temp
)

# 5. Verify the split sizes
print(f"Total rows: {len(df)}")
print(f"Training Set: {len(X_train)} rows ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation Set: {len(X_val)} rows ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test Set: {len(X_test)} rows ({len(X_test)/len(df)*100:.1f}%)")

Total rows: 4909
Training Set: 3435 rows (70.0%)
Validation Set: 737 rows (15.0%)
Test Set: 737 rows (15.0%)


## 2. Train on all the following models:

Logistic Regression

Decision Tree Classifier

Random Forest Classifier

Gradient Boosting Classifier (e.g., XGBoost or LightGBM)

K-Nearest Neighbors Classifier

Support Vector Classifier (SVC3)

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 1. Identify categorical and numerical features
categorical_features = [
    'gender',
    'ever_married',
    'work_type',
    'Residence_type',
    'smoking_status',
]
numerical_features = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']

# 2. Build preprocessing steps
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

# 3. Define the 6 required classification models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree Classifier': DecisionTreeClassifier(random_state=42),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors Classifier': KNeighborsClassifier(),
    'Support Vector Classifier (SVC)': SVC(probability=True, random_state=42),
}

# 4. Train each model using an ImbPipeline that applies SMOTE ONLY on training data
trained_models = {}
val_results = {}

print('--- Validation Performance (With Training SMOTE) ---')
for name, model in models.items():
  # ImbPipeline ensures SMOTE runs only when .fit() is called on X_train/y_train
  clf_pipeline = ImbPipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('smote', SMOTE(random_state=42)),
          ('classifier', model),
      ]
  )

  # Train the model
  clf_pipeline.fit(X_train, y_train)
  trained_models[name] = clf_pipeline

  # Predict on Validation set (Untouched by SMOTE)
  y_val_pred = clf_pipeline.predict(X_val)
  y_val_proba = clf_pipeline.predict_proba(X_val)[:, 1]

  acc = accuracy_score(y_val, y_val_pred)
  f1 = f1_score(y_val, y_val_pred, zero_division=0)
  try:
    roc_auc = roc_auc_score(y_val, y_val_proba)
  except:
    roc_auc = 0.0

  val_results[name] = {'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': roc_auc}
  print(
      f'{name} -> Accuracy: {acc:.4f} | F1-Score: {f1:.4f} | ROC-AUC:'
      f' {roc_auc:.4f}'
  )

--- Validation Performance (With Training SMOTE) ---
Logistic Regression -> Accuracy: 0.7585 | F1-Score: 0.2193 | ROC-AUC: 0.8477
Decision Tree Classifier -> Accuracy: 0.8901 | F1-Score: 0.0899 | ROC-AUC: 0.5263
Random Forest Classifier -> Accuracy: 0.9389 | F1-Score: 0.1509 | ROC-AUC: 0.7385
Gradient Boosting Classifier -> Accuracy: 0.8982 | F1-Score: 0.1935 | ROC-AUC: 0.8015
K-Nearest Neighbors Classifier -> Accuracy: 0.8331 | F1-Score: 0.2065 | ROC-AUC: 0.7383
Support Vector Classifier (SVC) -> Accuracy: 0.8155 | F1-Score: 0.1707 | ROC-AUC: 0.7767


# Validate and Compare all models:
Accuracy

Precision, Recall, F1-Score

ROC-AUC (if binary classification)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Initialize dictionaries to store performance results
evaluation_results = []

print('--- Evaluating All Models on Validation & Test Sets ---')

for name, pipeline in trained_models.items():
  # 1. Validation Set Metrics
  y_val_pred = pipeline.predict(X_val)
  y_val_proba = pipeline.predict_proba(X_val)[:, 1]

  val_acc = accuracy_score(y_val, y_val_pred)
  val_prec = precision_score(y_val, y_val_pred, zero_division=0)
  val_rec = recall_score(y_val, y_val_pred, zero_division=0)
  val_f1 = f1_score(y_val, y_val_pred, zero_division=0)
  try:
    val_auc = roc_auc_score(y_val, y_val_proba)
  except:
    val_auc = 0.0

  # 2. Test Set Metrics
  y_test_pred = pipeline.predict(X_test)
  y_test_proba = pipeline.predict_proba(X_test)[:, 1]

  test_acc = accuracy_score(y_test, y_test_pred)
  test_prec = precision_score(y_test, y_test_pred, zero_division=0)
  test_rec = recall_score(y_test, y_test_pred, zero_division=0)
  test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
  try:
    test_auc = roc_auc_score(y_test, y_test_proba)
  except:
    test_auc = 0.0

  # Append results
  evaluation_results.append({
      'Model': name,
      'Val Accuracy': val_acc,
      'Val Precision': val_prec,
      'Val Recall': val_rec,
      'Val F1-Score': val_f1,
      'Val ROC-AUC': val_auc,
      'Test Accuracy': test_acc,
      'Test Precision': test_prec,
      'Test Recall': test_rec,
      'Test F1-Score': test_f1,
      'Test ROC-AUC': test_auc,
  })

# 3. Create a clean comparison DataFrame
comparison_df = pd.DataFrame(evaluation_results)

# Display the formatted table
display(comparison_df.style.highlight_max(axis=0, color='lightgreen'))

--- Evaluating All Models on Validation & Test Sets ---


,Model,Val Accuracy,Val Precision,Val Recall,Val F1-Score,Val ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1-Score,Test ROC-AUC
0,Logistic Regression,0.758480,0.126904,0.806452,0.219298,0.847711,0.750339,0.119403,0.774194,0.206897,0.821804
1,Decision Tree Classifier,0.890095,0.068966,0.129032,0.089888,0.526273,0.895522,0.089286,0.161290,0.114943,0.544526
2,Random Forest Classifier,0.938942,0.181818,0.129032,0.150943,0.738486,0.932157,0.047619,0.032258,0.038462,0.750137
3,Gradient Boosting Classifier,0.898236,0.145161,0.290323,0.193548,0.801471,0.905020,0.117647,0.193548,0.146341,0.782966
4,K-Nearest Neighbors Classifier,0.833107,0.129032,0.516129,0.206452,0.738303,0.850746,0.108911,0.354839,0.166667,0.694599
5,Support Vector Classifier (SVC),0.815468,0.105263,0.451613,0.170732,0.776729,0.815468,0.105263,0.451613,0.170732,0.771726


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# 1. Select the Top 3 Models based on Validation ROC-AUC / F1-Score
# Based on your evaluation results, the top models are Logistic Regression, Gradient Boosting, and Random Forest
top_model_names = [
    'Logistic Regression',
    'Gradient Boosting Classifier',
    'Random Forest Classifier',
]
print(f'Top 3 Models selected for Ensemble: {top_model_names}')

# 2. Averaged Probability Ensemble
def get_ensemble_probabilities(X_data):
  # Collect predicted probabilities for the positive class (stroke=1) from the top 3 models
  probs = [
      trained_models[name].predict_proba(X_data)[:, 1] for name in top_model_names
  ]
  # Average the probabilities across the 3 models
  return np.mean(probs, axis=0)


# Validation Averaged Predictions
val_avg_proba = get_ensemble_probabilities(X_val)
val_avg_pred = (val_avg_proba >= 0.5).astype(int)

# Test Averaged Predictions
test_avg_proba = get_ensemble_probabilities(X_test)
test_avg_pred = (test_avg_proba >= 0.5).astype(int)


# 3. Bayesian Ensemble Approximation (Weighted average based on Validation ROC-AUC performance)
# Higher performing models receive a proportionally higher Bayesian/confidence weight
val_auc_scores = [val_results[name]['ROC-AUC'] for name in top_model_names]
weights = np.array(val_auc_scores) / np.sum(val_auc_scores)
print(f'Bayesian Ensemble Weights assigned to top 3 models: {weights}')


def get_bayesian_ensemble_probabilities(X_data):
  probs = [
      trained_models[name].predict_proba(X_data)[:, 1] for name in top_model_names
  ]
  # Compute weighted average probabilities
  return np.average(probs, axis=0, weights=weights)


# Validation Bayesian Predictions
val_bayes_proba = get_bayesian_ensemble_probabilities(X_val)
val_bayes_pred = (val_bayes_proba >= 0.5).astype(int)

# Test Bayesian Predictions
test_bayes_proba = get_bayesian_ensemble_probabilities(X_test)
test_bayes_pred = (test_bayes_proba >= 0.5).astype(int)


# 4. Evaluate and Compare Both Ensembles
def evaluate_preds(y_true, y_pred, y_proba, model_name):
  return {
      'Model': model_name,
      'Accuracy': accuracy_score(y_true, y_pred),
      'Precision': precision_score(y_true, y_pred, zero_division=0),
      'Recall': recall_score(y_true, y_pred, zero_division=0),
      'F1-Score': f1_score(y_true, y_pred, zero_division=0),
      'ROC-AUC': roc_auc_score(y_true, y_proba),
  }


ensemble_comparison = [
    evaluate_preds(
        val_arg := y_val, val_avg_pred, val_avg_proba, 'Validation - Averaged Ensemble'
    ),
    evaluate_preds(
        y_val, val_bayes_pred, val_bayes_proba, 'Validation - Bayesian Ensemble'
    ),
    evaluate_preds(
        y_test, test_avg_pred, test_avg_proba, 'Test - Averaged Ensemble'
    ),
    evaluate_preds(
        y_test, test_bayes_pred, test_bayes_proba, 'Test - Bayesian Ensemble'
    ),
]

ensemble_df = pd.DataFrame(ensemble_comparison)
display(ensemble_df.style.highlight_max(axis=0, color='lightgreen'))

Top 3 Models selected for Ensemble: ['Logistic Regression', 'Gradient Boosting Classifier', 'Random Forest Classifier']
Bayesian Ensemble Weights assigned to top 3 models: [0.35503717 0.33567116 0.30929167]


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Validation - Averaged Ensemble,0.896879,0.142857,0.290323,0.191489,0.816869
1,Validation - Bayesian Ensemble,0.886024,0.136986,0.322581,0.192308,0.818834
2,Test - Averaged Ensemble,0.900950,0.125000,0.225806,0.160920,0.806634
3,Test - Bayesian Ensemble,0.892809,0.112903,0.225806,0.150538,0.807594


In [1]:
import pandas as pd

# 1. Compile your evaluation results into a DataFrame
comparison_df = pd.DataFrame(evaluation_results)

# 2. Display the table directly in your notebook
display(comparison_df)

# 3. Export the table to a CSV file if needed
comparison_df.to_csv('Model_Comparison_Table.csv', index=False)

NameError: name 'evaluation_results' is not defined